<a href="https://colab.research.google.com/github/jv48391592-eng/PIPELINE-RAG-COM-FINE-TUNING-LORA-E-DISPONIBILIZA-O-VIA-API-RESTFUL/blob/main/notebook/rag.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#CÉLULA 1 — INSTALAÇÃO

!pip install -q \
transformers \
accelerate \
bitsandbytes \
datasets \
sentencepiece \
peft \
trl \
pdfplumber \
sacrebleu \
rouge-score \
fastapi \
uvicorn \
pyngrok \
matplotlib \
pandas \
scikit-learn

In [ ]:
#CÉLULA 2 — GERAÇÃO DO DATASET (ETAPA 1)

import json
from transformers import pipeline

INPUT_TXT = "20260050979.txt"

OUTPUT_DATASET_BRUTO = "dataset_gerado_bruto.jsonl"

CHUNK_SIZES = [500, 900]

OVERLAP = 120

GENERATOR_MODEL = "Qwen/Qwen2-1.5B-Instruct"

PROMPT_TEMPLATE = """
Você é um gerador de datasets para fine-tuning de modelos de linguagem.

Baseado APENAS no texto abaixo, gere UM único par pergunta/resposta.

REGRAS:
- Não invente informações
- A pergunta deve ser clara e objetiva
- A resposta deve ser fiel ao texto
- Não use conhecimento externo
- Evite perguntas genéricas
- A resposta deve possuir entre 30 e 120 palavras

Texto:
{chunk}

Formato obrigatório:

Instruction: [pergunta]
Output: [resposta]
"""

def carregar_texto(caminho):

    with open(caminho, "r", encoding="utf-8") as f:
        return f.read()

def criar_chunks(texto, tamanho, overlap):

    chunks = []

    start = 0

    while start < len(texto):

        end = start + tamanho

        chunk = texto[start:end]

        chunks.append(chunk)

        start = end - overlap

        if start >= len(texto):
            break

    return chunks

def gerar_par(chunk, generator):

    prompt = PROMPT_TEMPLATE.format(chunk=chunk)

    try:

        output = generator(
            prompt,
            max_new_tokens=180,
            do_sample=False
        )[0]["generated_text"]

        if "Instruction:" in output and "Output:" in output:

            inst = output.split("Instruction:")[1].split("Output:")[0].strip()

            out = output.split("Output:")[1].strip()

            return {
                "Instruction": inst,
                "Output": out
            }

    except Exception as e:
        print("Erro:", e)

    return None

texto = carregar_texto(INPUT_TXT)

print(f"Texto carregado: {len(texto)} caracteres")

todos_pares = []

generator = pipeline(
    "text-generation",
    model=GENERATOR_MODEL,
    device_map="auto"
)

for tamanho in CHUNK_SIZES:

    print(f"\n=== Chunk Size: {tamanho} ===")

    chunks = criar_chunks(
        texto,
        tamanho,
        OVERLAP
    )

    print(f"Chunks gerados: {len(chunks)}")

    pares_gerados = 0

    for chunk in chunks[:150]:

        par = gerar_par(chunk, generator)

        if par:

            todos_pares.append(par)

            pares_gerados += 1

    print(f"Pares válidos: {pares_gerados}")

with open(OUTPUT_DATASET_BRUTO, "w", encoding="utf-8") as f:

    for item in todos_pares:

        f.write(
            json.dumps(item, ensure_ascii=False) + "\n"
        )

print(f"\nDataset bruto salvo em: {OUTPUT_DATASET_BRUTO}")

print(f"Total de pares: {len(todos_pares)}")

In [ ]:
 #CÉLULA 3 — CURADORIA AUTOMÁTICA

import json

INPUT_FILE = "dataset_gerado_bruto.jsonl"

OUTPUT_FILE = "dataset_limpo.jsonl"

dados_limpos = []

removidos = 0

with open(INPUT_FILE, "r", encoding="utf-8") as f:

    for linha in f:

        item = json.loads(linha)

        inst = item.get("Instruction", "").strip()

        out = item.get("Output", "").strip()

        if len(inst) < 10:
            removidos += 1
            continue

        if len(out) < 30:
            removidos += 1
            continue

        if "não sei" in out.lower():
            removidos += 1
            continue

        if "não encontrado" in out.lower():
            removidos += 1
            continue

        dados_limpos.append({
            "Instruction": inst,
            "Output": out
        })

with open(OUTPUT_FILE, "w", encoding="utf-8") as f:

    for item in dados_limpos:

        f.write(
            json.dumps(item, ensure_ascii=False) + "\n"
        )

print(f"Total válidos: {len(dados_limpos)}")

print(f"Removidos: {removidos}")

print(
    f"Percentual removido: "
    f"{(removidos/(len(dados_limpos)+removidos))*100:.2f}%"
)

In [ ]:
#CÉLULA 4 — TREINAMENTO LoRA COMPLETO

!pip install "torchao>=0.16.0"

from transformers import (
    AutoModelForCausalLM,
    AutoModelForSeq2SeqLM,
    AutoTokenizer,
    Trainer,
    TrainingArguments
)

from peft import (
    LoraConfig,
    get_peft_model
)

from datasets import load_dataset

import matplotlib.pyplot as plt

dataset = load_dataset(
    "json",
    data_files="dataset_limpo.jsonl"
)["train"]

def normalize_columns(example):

    example["instruction"] = (
        example.get("instruction")
        or example.get("Instruction")
        or ""
    )

    example["output"] = (
        example.get("output")
        or example.get("Output")
        or ""
    )

    return example

dataset = dataset.map(normalize_columns)

def tokenize_causal(example, tokenizer, max_length=256):

    text = (
        f"Instruction: {example['instruction']}\n\n"
        f"Output: {example['output']}"
    )

    tokens = tokenizer(
        text,
        truncation=True,
        padding="max_length",
        max_length=max_length
    )

    tokens["labels"] = tokens["input_ids"].copy()

    return tokens

def tokenize_seq2seq(example, tokenizer, max_length=256):

    input_text = f"Instruction: {example['instruction']}"

    target_text = example["output"]

    model_inputs = tokenizer(
        input_text,
        truncation=True,
        padding="max_length",
        max_length=max_length
    )

    labels = tokenizer(
        target_text,
        truncation=True,
        padding="max_length",
        max_length=max_length
    )

    model_inputs["labels"] = labels["input_ids"]

    return model_inputs

def get_lora_config(name):

    if "gpt" in name or "opt" in name:

        return LoraConfig(
            r=16,
            lora_alpha=32,
            lora_dropout=0.1,
            bias="none",
            task_type="CAUSAL_LM",
            target_modules=[
                "q_proj",
                "v_proj",
                "k_proj",
                "out_proj"
            ]
        )

    elif "t5" in name:

        return LoraConfig(
            r=16,
            lora_alpha=32,
            lora_dropout=0.1,
            bias="none",
            task_type="SEQ_2_SEQ_LM",
            target_modules=[
                "q",
                "k",
                "v",
                "o"
            ]
        )

    elif "bart" in name:

        return LoraConfig(
            r=16,
            lora_alpha=32,
            lora_dropout=0.1,
            bias="none",
            task_type="SEQ_2_SEQ_LM",
            target_modules=[
                "q_proj",
                "k_proj",
                "v_proj",
                "out_proj"
            ]
        )

models_to_train = {

    "gpt-neo-125M": {
        "base": "EleutherAI/gpt-neo-125M",
        "save_dir": "lora_gptneo"
    },

    "opt-125M": {
        "base": "facebook/opt-125m",
        "save_dir": "lora_opt125"
    },

    "t5-small": {
        "base": "t5-small",
        "save_dir": "lora_t5"
    },

    "bart-base": {
        "base": "facebook/bart-base",
        "save_dir": "lora_bart"
    }
}

for name, info in models_to_train.items():

    print(f"\nTreinando {name}")

    tokenizer = AutoTokenizer.from_pretrained(info["base"])

    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    if "t5" in name or "bart" in name:

        model = AutoModelForSeq2SeqLM.from_pretrained(
            info["base"],
            device_map="auto"
        )

        tokenize_fn = lambda ex: tokenize_seq2seq(ex, tokenizer)

    else:

        model = AutoModelForCausalLM.from_pretrained(
            info["base"],
            device_map="auto"
        )

        tokenize_fn = lambda ex: tokenize_causal(ex, tokenizer)

    tokenized = dataset.map(tokenize_fn)

    lora_config = get_lora_config(name)

    model = get_peft_model(model, lora_config)

    model.print_trainable_parameters()

    training_args = TrainingArguments(
        output_dir=f"./results_{name}",
        per_device_train_batch_size=4,
        num_train_epochs=3,
        learning_rate=2e-4,
        logging_steps=20,
        save_strategy="no",
        report_to="none",
        remove_unused_columns=False
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=tokenized
    )

    trainer.train()

    model.save_pretrained(info["save_dir"])

    tokenizer.save_pretrained(info["save_dir"])

    print(f"Modelo salvo em: {info['save_dir']}")

    loss_history = []

    for log in trainer.state.log_history:

        if "loss" in log:
            loss_history.append(log["loss"])

    plt.figure(figsize=(8,5))

    plt.plot(loss_history)

    plt.title(f"Loss - {name}")

    plt.xlabel("Steps")

    plt.ylabel("Loss")

    plt.grid(True)

    plt.show()

In [ ]:
#CÉLULA 5 — AVALIAÇÃO COMPLETA

!pip install rouge_score
!pip install sacrebleu

from datasets import load_dataset

from transformers import (
    pipeline,
    AutoTokenizer,
    AutoModelForCausalLM,
    AutoModelForSeq2SeqLM
)

from peft import PeftModel

from rouge_score import rouge_scorer

from sacrebleu import sentence_bleu

import numpy as np

import pandas as pd

import matplotlib.pyplot as plt

import torch

import math

dataset = load_dataset(
    "json",
    data_files="dataset_limpo.jsonl"
)["train"]

test_examples = dataset.select(range(50))

def normalize(example):

    example["instruction"] = (
        example.get("instruction")
        or example.get("Instruction")
        or ""
    )

    example["output"] = (
        example.get("output")
        or example.get("Output")
        or ""
    )

    return example

test_examples = test_examples.map(normalize)

def load_model(save_dir, base_name, is_seq2seq=False):

    tokenizer = AutoTokenizer.from_pretrained(base_name)

    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    if is_seq2seq:

        base = AutoModelForSeq2SeqLM.from_pretrained(
            base_name,
            device_map="auto"
        )

    else:

        base = AutoModelForCausalLM.from_pretrained(
            base_name,
            device_map="auto"
        )

    model = PeftModel.from_pretrained(base, save_dir)

    model.eval()

    return model, tokenizer, is_seq2seq

models = {

    "gpt-neo": load_model(
        "lora_gptneo",
        "EleutherAI/gpt-neo-125M",
        False
    ),

    "opt": load_model(
        "lora_opt125",
        "facebook/opt-125m",
        False
    ),

    "t5": load_model(
        "lora_t5",
        "t5-small",
        True
    ),

    "bart": load_model(
        "lora_bart",
        "facebook/bart-base",
        True
    )
}

def generate_response(model_name, instruction, max_new=150):

    model, tokenizer, is_seq2seq = models[model_name]

    if is_seq2seq:

        inputs = tokenizer(
            f"Instruction: {instruction}",
            return_tensors="pt"
        ).to(model.device)

        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new,
            temperature=0.7,
            do_sample=True
        )

        return tokenizer.decode(
            outputs[0],
            skip_special_tokens=True
        )

    else:

        pipe = pipeline(
            "text-generation",
            model=model,
            tokenizer=tokenizer
        )

        result = pipe(
            f"Instruction: {instruction}\n\nOutput:",
            max_new_tokens=max_new
        )[0]["generated_text"]

        return result.split("Output:")[-1].strip()

def calculate_perplexity(model, tokenizer, text):

    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=256
    ).to(model.device)

    with torch.no_grad():

        outputs = model(
            **inputs,
            labels=inputs["input_ids"]
        )

    loss = outputs.loss

    return math.exp(loss.item())

def jaccard_similarity(a, b):

    set1 = set(a.lower().split())

    set2 = set(b.lower().split())

    inter = len(set1.intersection(set2))

    union = len(set1.union(set2))

    return inter / union if union > 0 else 0

def faithfulness(reference, generated):

    ref_words = set(reference.lower().split())

    gen_words = set(generated.lower().split())

    overlap = len(ref_words.intersection(gen_words))

    return overlap / len(gen_words) if len(gen_words) > 0 else 0

def plan_adherence(reference, generated):

    ref_has_list = "-" in reference or "1." in reference

    gen_has_list = "-" in generated or "1." in generated

    return 1 if ref_has_list == gen_has_list else 0

scorer = rouge_scorer.RougeScorer(
    ['rouge1', 'rouge2', 'rougeL'],
    use_stemmer=True
)

results = {}

for model_name in models.keys():

    print(f"\nAvaliando {model_name}")

    rouge1 = []

    rouge2 = []

    rougeL = []

    bleu_scores = []

    ppl_scores = []

    faith_scores = []

    relevance_scores = []

    plan_scores = []

    for ex in test_examples:

        instruction = ex["instruction"]

        reference = ex["output"]

        generated = generate_response(
            model_name,
            instruction
        )

        scores = scorer.score(reference, generated)

        rouge1.append(scores["rouge1"].fmeasure)

        rouge2.append(scores["rouge2"].fmeasure)

        rougeL.append(scores["rougeL"].fmeasure)

        bleu_scores.append(
            sentence_bleu(
                generated,
                [reference]
            ).score
        )

        ppl_scores.append(
            calculate_perplexity(
                models[model_name][0],
                models[model_name][1],
                f"{instruction} {reference}"
            )
        )

        faith_scores.append(
            faithfulness(reference, generated)
        )

        relevance_scores.append(
            jaccard_similarity(
                instruction,
                generated
            )
        )

        plan_scores.append(
            plan_adherence(
                reference,
                generated
            )
        )

    results[model_name] = {

        "ROUGE-1": np.mean(rouge1),

        "ROUGE-2": np.mean(rouge2),

        "ROUGE-L": np.mean(rougeL),

        "BLEU": np.mean(bleu_scores),

        "Perplexity": np.mean(ppl_scores),

        "Faithfulness": np.mean(faith_scores),

        "Answer_Relevance": np.mean(relevance_scores),

        "Plan_Adherence": np.mean(plan_scores)
    }

df = pd.DataFrame(results).T

print(df)

metrics = [
    "ROUGE-1",
    "ROUGE-2",
    "ROUGE-L",
    "BLEU",
    "Faithfulness",
    "Answer_Relevance",
    "Plan_Adherence"
]

angles = np.linspace(
    0,
    2*np.pi,
    len(metrics),
    endpoint=False
).tolist()

angles += angles[:1]

fig, ax = plt.subplots(
    figsize=(8,8),
    subplot_kw=dict(polar=True)
)

for model_name in df.index:

    values = df.loc[model_name, metrics].tolist()

    values += values[:1]

    ax.plot(angles, values, label=model_name)

    ax.fill(angles, values, alpha=0.1)

ax.set_xticks(angles[:-1])

ax.set_xticklabels(metrics)

plt.legend(loc="upper right")

plt.title("Radar de Métricas")

plt.show()

print("\nANÁLISE QUALITATIVA")

for ex in test_examples.select(range(3)):

    print("\n" + "="*80)

    print("INSTRUÇÃO:")
    print(ex["instruction"])

    print("\nREFERÊNCIA:")
    print(ex["output"])

    for model_name in models.keys():

        resposta = generate_response(
            model_name,
            ex["instruction"]
        )

        print(f"\n[{model_name.upper()}]")
        print(resposta)

In [ ]:
#CÉLULA 6 — API FASTAPI COMPLETA

# =========================================================
# INSTALAÇÃO
# =========================================================

!pip uninstall -y uvicorn -q

!pip install -q \
uvicorn==0.29.0 \
fastapi \
pyngrok \
peft \
transformers \
accelerate \
torch \
nest_asyncio

# =========================================================
# IMPORTS
# =========================================================

import nest_asyncio
nest_asyncio.apply()

import asyncio
import threading
import time

from contextlib import asynccontextmanager

from fastapi import FastAPI
from pydantic import BaseModel

from pyngrok import ngrok

import uvicorn

from transformers import (
    pipeline,
    AutoTokenizer,
    AutoModelForCausalLM,
    AutoModelForSeq2SeqLM
)

from peft import PeftModel

# =========================================================
# LOAD MODELOS CAUSAIS
# =========================================================

def load_causal(save_dir, base_model):

    tokenizer = AutoTokenizer.from_pretrained(base_model)

    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    base = AutoModelForCausalLM.from_pretrained(
        base_model,
        device_map="auto"
    )

    model = PeftModel.from_pretrained(
        base,
        save_dir
    )

    model.eval()

    pipe = pipeline(
        "text-generation",
        model=model,
        tokenizer=tokenizer
    )

    return pipe

# =========================================================
# LOAD MODELOS SEQ2SEQ
# =========================================================

def load_seq2seq(save_dir, base_model):

    tokenizer = AutoTokenizer.from_pretrained(base_model)

    model_base = AutoModelForSeq2SeqLM.from_pretrained(
        base_model,
        device_map="auto"
    )

    model = PeftModel.from_pretrained(
        model_base,
        save_dir
    )

    model.eval()

    return model, tokenizer

# =========================================================
# MODELOS
# =========================================================

MODELS = {}

@asynccontextmanager
async def lifespan(app: FastAPI):

    global MODELS

    print("Carregando modelos...")

    MODELS = {

        "gpt-neo": load_causal(
            "lora_gptneo",
            "EleutherAI/gpt-neo-125M"
        ),

        "opt": load_causal(
            "lora_opt125",
            "facebook/opt-125m"
        ),

        "t5": load_seq2seq(
            "lora_t5",
            "t5-small"
        ),

        "bart": load_seq2seq(
            "lora_bart",
            "facebook/bart-base"
        )
    }

    print("✅ 4 modelos carregados!")

    yield

# =========================================================
# FASTAPI
# =========================================================

app = FastAPI(
    title="RAG API",
    lifespan=lifespan
)

# =========================================================
# REQUEST
# =========================================================

class ChatRequest(BaseModel):

    modelo: str
    pergunta: str

# =========================================================
# ROOT
# =========================================================

@app.get("/")
async def root():

    return {
        "status": "API ONLINE"
    }

# =========================================================
# HEALTH
# =========================================================

@app.get("/health")
async def health():

    return {
        "status": "online",
        "modelos": list(MODELS.keys())
    }

# =========================================================
# MODELOS
# =========================================================

@app.get("/modelos")
async def modelos():

    return [

        {
            "id": "gpt-neo",
            "nome": "GPT-Neo Fine-Tuned",
            "descricao": "Modelo causal com LoRA"
        },

        {
            "id": "opt",
            "nome": "OPT Fine-Tuned",
            "descricao": "Modelo causal com LoRA"
        },

        {
            "id": "t5",
            "nome": "T5 Fine-Tuned",
            "descricao": "Modelo Seq2Seq com LoRA"
        },

        {
            "id": "bart",
            "nome": "BART Fine-Tuned",
            "descricao": "Modelo Seq2Seq com LoRA"
        }
    ]

# =========================================================
# CHAT
# =========================================================

@app.post("/chat")
async def chat(req: ChatRequest):

    prompt = f"""
Instruction: {req.pergunta}
Output:
"""

    # =========================
    # CAUSAIS
    # =========================

    if req.modelo in ["gpt-neo", "opt"]:

        pipe = MODELS[req.modelo]

        result = pipe(
            prompt,
            max_new_tokens=180,
            do_sample=True,
            temperature=0.7
        )[0]["generated_text"]

        resposta = result.split("Output:")[-1].strip()

    # =========================
    # SEQ2SEQ
    # =========================

    else:

        model, tokenizer = MODELS[req.modelo]

        inputs = tokenizer(
            prompt,
            return_tensors="pt"
        ).to(model.device)

        outputs = model.generate(
            **inputs,
            max_new_tokens=180
        )

        resposta = tokenizer.decode(
            outputs[0],
            skip_special_tokens=True
        )

    return {
        "modelo": req.modelo,
        "pergunta": req.pergunta,
        "resposta": resposta
    }

# =========================================================
# EXECUTAR UVICORN SEM BUG DO COLAB
# =========================================================

def run_server():

    config = uvicorn.Config(
        app=app,
        host="0.0.0.0",
        port=8000,
        loop="asyncio"
    )

    server = uvicorn.Server(config)

    asyncio.set_event_loop(asyncio.new_event_loop())

    server.run()

thread = threading.Thread(
    target=run_server,
    daemon=True
)

thread.start()

# aguarda iniciar
time.sleep(8)

# =========================================================
# NGROK
# =========================================================

NGROK_TOKEN = "insira sua key"

ngrok.set_auth_token(NGROK_TOKEN)

public_url = ngrok.connect(8000)

print("\n🌐 URL Pública:")
print(public_url)

print("\n✅ API pronta!")
print(f"{public_url}/docs")